# ch12 类别不平衡：class_weight / 阈值移动 / SMOTE

> 数据：`defect_ml.csv`，正例占 21.3%（危急缺陷）。不平衡数据的三个处方：
> **改损失（class_weight）、改判决（阈值移动）、改数据（SMOTE）**。
>
> 竞赛评分点「模型性能评估」里，不平衡数据用错指标 = 白干。

**本章考点**

1. 准确率的遮羞布：基线 0.7833 vs 召回 0.1923
2. `class_weight="balanced"` 与自定义权重的隐式权重公式
3. 阈值移动与 class_weight 的等价性
4. SMOTE 的正确姿势：只碰训练集
5. 测试集重采样 = 作弊
6. `imblearn.Pipeline` 防 CV 泄漏

**难点索引**

| 难点 | 为什么是坑 | 位置 |
|---|---|---|
| acc 遮羞布 | 21.3% 正例率白送的分 | §12.1 |
| balanced 真身 | 隐式权重 n/(2·类计数)，≈ 阈值移动 | §12.2/12.4 |
| SMOTE 泄漏 | 先重采样后切折 → 验证折混入合成样本 | §12.6 |
| 测试集重采样 | 指标全部作废 | §12.5 |

In [ ]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")

In [ ]:
# 数据与划分口径沿用 ch05：分层 8:2、median 填充、独热、标准化
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y_all = dev[TARGET]

tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y_all)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
feat = NUM + [f"{c}_{v}" for c, vals in zip(CAT, oh.categories_) for v in vals]
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("sklearn", sklearn.__version__, "| imblearn",
      __import__("imblearn").__version__)
print("训练正例:", int(ys_tr.sum()), "/", len(ys_tr),
      "| 测试正例:", int(ys_te.sum()), "/", len(ys_te))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def report_cm(tag, yt, yp):
    """一行报 acc/P/R/f1 与混淆矩阵，返回 (acc, recall, f1)。"""
    acc = round(float(accuracy_score(yt, yp)), 4)
    rec = round(float(recall_score(yt, yp)), 4)
    f1v = round(float(f1_score(yt, yp, zero_division=0)), 4)
    print(f"{tag}: acc={acc} P={round(float(precision_score(yt, yp, zero_division=0)), 4)} "
          f"R={rec} f1={f1v} cm={confusion_matrix(yt, yp).tolist()}")
    return acc, rec, f1v


print("脚手架就绪：report_cm")

## 12.1 准确率的遮羞布

基线 LR acc 0.7833，看着比「全判负躺平」的 0.7867 差不太多——因为
21.3% 的正例率本来就白送。拆开看：26 个真危急只抓住 5 个（recall 0.1923），
f1 只有 0.2778。**不平衡数据先报 recall/f1，acc 是最不诚实的指标**。

In [ ]:
# 基线体检：0.7833 的准确率是什么成色
base = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ys_tr)
base_acc, base_rec, base_f1 = report_cm("基线 thr=0.5", ys_te, base.predict(Xte_s))
print("全判负的躺平分:", round(float(1 - ys_te.mean()), 4))
print("解读: 模型只比躺平多抓 5 个正例，代价是漏 21 个——不平衡数据"
      "先看 recall/f1 再看 acc，否则 21.3% 的正例率就是你白捡的分")

assert (base_acc, base_rec, base_f1) == (0.7833, 0.1923, 0.2778)

## 12.2 难点深挖：class_weight 的真身

`class_weight="balanced"` 的隐式权重 = n/(2·类计数)：本数据约
负类 0.55 / 正类 2.35——正例错一次罚 4 倍多。效果：FN 21 → 5，
recall 0.1923 → 0.8077，代价是 acc 掉 8 个点。自定义字典
`{0: 1, 1: 4}` 更激进（recall 0.8462），适合把业务代价直接写进去。

In [ ]:
# class_weight=balanced：损失函数里反加权（不改数据）
cw = LogisticRegression(max_iter=1000, random_state=42,
                        class_weight="balanced").fit(Xtr_s, ys_tr)
report_cm("balanced", ys_te, cw.predict(Xte_s))
print("解读: FN 21 → 5。balanced 的隐式权重 = n/(2*类计数)：负类 ~0.55、正类 ~2.35，"
      "正例错一次罚 4 倍多。零成本方案，竞赛首选")

assert round(float(accuracy_score(ys_te, cw.predict(Xte_s))), 4) == 0.7
assert round(float(recall_score(ys_te, cw.predict(Xte_s))), 4) == 0.8077

In [ ]:
# 自定义权重：业务代价直接写进字典
cw4 = LogisticRegression(max_iter=1000, random_state=42,
                         class_weight={0: 1, 1: 4}).fit(Xtr_s, ys_tr)
report_cm("w={0:1,1:4}", ys_te, cw4.predict(Xte_s))
print("解读: 权重比 = 漏报代价比。变压器危急缺陷漏一次的代价远高于虚惊，"
      "给 1:4 甚至 1:10 都合理——把业务对话翻译成字典就交差了")

assert round(float(recall_score(ys_te, cw4.predict(Xte_s))), 4) == 0.8462

## 12.3 阈值移动：不重训，只动判决线

阈值 0.5 → 0.3：recall 0.1923 → 0.6538，f1 0.5397 反超 balanced。
**阈值 0.2 的混淆矩阵与 balanced 完全一致**——class_weight 本质就是
训练时的阈值移动，两者二选一。阈值移动的额外好处：不重训模型，
交卷时按指标现调。

In [ ]:
# 阈值移动：模型不动，判决线动
p_te = base.predict_proba(Xte_s)[:, 1]
for t in (0.5, 0.4, 0.3, 0.2, 0.15):
    report_cm(f"阈值 {t}", ys_te, (p_te >= t).astype(int))
print("阈值 0.2 的 cm == balanced 的 cm:",
      (confusion_matrix(ys_te, (p_te >= 0.2).astype(int)).tolist()
       == confusion_matrix(ys_te, cw.predict(Xte_s)).tolist()))
print("解读: 降阈值 = 越敢报正例。class_weight 本质就是『训练时的阈值移动』——"
      "两者二选一即可；阈值移动还有个好处：不用重训模型，交卷时按指标现调")

assert (confusion_matrix(ys_te, (p_te >= 0.2).astype(int)).tolist()
        == confusion_matrix(ys_te, cw.predict(Xte_s)).tolist())

## 12.4 SMOTE：改数据

SMOTE 在少数类样本之间线性插值：训练集 480 → 756（正例 102 → 378，
补到 1:1）。测试集成绩与 balanced 同档（recall 0.7692），但多了一层
随机性和 k_neighbors 超参。**铁律：只碰训练集**——测试集重采样
（120 → 188）后所有指标作废，那是人工改题。

In [ ]:
# SMOTE：在少数类样本之间线性插值造新样本（只动训练集！）
sm = SMOTE(random_state=42)
Xtr_sm, ys_tr_sm = sm.fit_resample(Xtr_s, ys_tr)
lr_sm = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_sm, ys_tr_sm)
report_cm("SMOTE+LR", ys_te, lr_sm.predict(Xte_s))
print("SMOTE 前:", Xtr_s.shape, "正例", int(ys_tr.sum()), "→ 后:", Xtr_sm.shape,
      "正例", int(ys_tr_sm.sum()))
print("解读: auto 策略补到 1:1（102→378，新增 276 个合成点）；效果与 balanced 同档，"
      "但随机性更大、多一个要调的超参（k_neighbors）")

assert Xtr_sm.shape == (756, 16) and int(ys_tr_sm.sum()) == 378
assert round(float(recall_score(ys_te, lr_sm.predict(Xte_s))), 4) == 0.7692

In [ ]:
# 错误示范：测试集 SMOTE = 人工改题
Xte_sm, ys_te_sm = SMOTE(random_state=42).fit_resample(Xte_s, ys_te)
bad_acc = round(float(accuracy_score(ys_te_sm, lr_sm.predict(Xte_sm))), 4)
print("作弊后的『acc』:", bad_acc, "| 真实 acc:",
      round(float(accuracy_score(ys_te, lr_sm.predict(Xte_s))), 4))
print("解读: 测试集 120 → 188（94:94），其中 68 个是合成的——指标再好看也是自娱自乐。"
      "铁律：**重采样永远只碰训练集**")

assert len(ys_te_sm) == 188 and len(ys_te) == 120, "测试集 120 → 188（94:94），已经不是同一道题了"

## 12.5 难点深挖：imblearn Pipeline 防 CV 泄漏

先在全体训练数据上 SMOTE 再切折 = 验证折里混入训练折的合成样本，
CV 分数虚高。`imblearn.pipeline.Pipeline` 把 SMOTE 塞进每折内部：
CV recall 0.6067 vs 普通 LR 0.3533，且这个差距是干净的。

In [ ]:
# imblearn Pipeline：CV 每折训练段内部重采样，验证折保持原味
imb_pipe = ImbPipeline([("smote", SMOTE(random_state=42)),
                        ("clf", LogisticRegression(max_iter=1000, random_state=42))])
sm_sc = cross_val_score(imb_pipe, Xtr_s, ys_tr, cv=skf, scoring="recall")
plain_sc = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                           Xtr_s, ys_tr, cv=skf, scoring="recall")
print("SMOTE 管道 CV recall:", np.round(sm_sc, 4).tolist(), "| mean", round(float(sm_sc.mean()), 4))
print("普通 LR  CV recall:", np.round(plain_sc, 4).tolist(), "| mean", round(float(plain_sc.mean()), 4))
print("解读: 如果先在全体训练数据上 SMOTE 再切折，验证折里就混进了训练折的合成样本——"
      "CV 分数虚高。imblearn 的 Pipeline 把 SMOTE 塞进每折内部，天然防泄漏")

assert round(float(sm_sc.mean()), 4) == 0.6067
assert round(float(plain_sc.mean()), 4) == 0.3533

## 12.6 三方案对表

| 方案 | acc | recall | f1 | 成本 |
|---|---|---|---|---|
| 基线 thr=0.5 | 0.7833 | 0.1923 | 0.2778 | 0 |
| balanced | 0.70 | 0.8077 | 0.5385 | 一个参数 |
| SMOTE | 0.6917 | 0.7692 | 0.5195 | 造样本+超参 |
| 阈值 0.3 | **0.7583** | 0.6538 | **0.5397** | 不重训 |

竞赛推荐顺序：**class_weight → 阈值移动 → SMOTE**。漏报代价大的业务
（变压器危急缺陷）保 recall；只要排序能力（AUC）则原模型 + 移动阈值。

In [ ]:
# 三方案对表：同一个测试集，三种打法
report_cm("基线 thr=0.5", ys_te, base.predict(Xte_s))
report_cm("balanced", ys_te, cw.predict(Xte_s))
report_cm("SMOTE", ys_te, lr_sm.predict(Xte_s))
report_cm("阈值 0.3", ys_te, (p_te >= 0.3).astype(int))
print("解读: 本数据（样本少、正例 21%）上：阈值移动 f1 最高且 acc 保得住；"
      "balanced 最省事；SMOTE 效果同档但更重。**竞赛推荐顺序：class_weight → "
      "阈值移动 → SMOTE**")

assert round(float(f1_score(ys_te, (p_te >= 0.3).astype(int), zero_division=0)), 4) == 0.5397

## 小结

1. 不平衡数据先报 recall/f1：基线 acc 0.7833 里藏着 FN=21。
2. `class_weight="balanced"` 隐式权重 n/(2·类计数)，零成本首选。
3. 自定义字典 = 业务代价直译；权重越大模型越敢报正例。
4. 阈值移动与 class_weight 等价（阈值 0.2 == balanced 的 cm）。
5. SMOTE 只碰训练集；测试集重采样 = 人工改题，指标作废。
6. CV 里必须用 `imblearn.Pipeline`，先重采样后切折是泄漏。
7. 选型：class_weight → 阈值移动 → SMOTE；本数据阈值移动 f1 最高。